In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from bm3d import bm3d
from tqdm import tqdm
from torchvision import transforms
import os
import pandas as pd

from HARUnet_model_v2_1 import HARU_net

from ResUNet_model import ResUNet


#from torch.cuda.amp import GradScaler, autocast
#from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure as ssim
from utils import psnr, batch_psnr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0):
        """
        Args:
            patience (int): How many epochs to wait after last improvement.
            delta (float): Minimum change to consider an improvement.
        """
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            print(f'Best loss = {self.best_loss}')
            self.counter = 0

In [2]:
# Load data
def load_data(pickle_file):
    with open(pickle_file, 'rb') as f:
        patches = pickle.load(f)
    return patches  # Expecting a NumPy array (N, 1, H, W)

class CBCTDataset(Dataset):
    def __init__(self, noisy_patches, target_patches):
        self.noisy = noisy_patches  # Keep as is
        self.target = target_patches  # Keep as is

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy_tensor = torch.tensor(self.noisy[idx], dtype=torch.float32)
        target_tensor = torch.tensor(self.target[idx], dtype=torch.float32)

        return noisy_tensor, target_tensor
    

In [4]:
pickled_train_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_noisyCBCT_patches.pkl"
pickled_train_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_origCBCT_patches.pkl"
train_inputs = load_data(pickled_train_inputs)  # Shape: (N, 1, H, W)
train_targets = load_data(pickled_train_targets)  # Shape: (N, 1, H, W)

train_dataset = CBCTDataset(train_inputs, train_targets)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

pickled_val_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_noisyCBCT_patches.pkl"
pickled_val_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_origCBCT_patches.pkl"
val_inputs = load_data(pickled_val_inputs)  # Shape: (N, 1, H, W)
val_targets = load_data(pickled_val_targets)  # Shape: (N, 1, H, W)

val_dataset = CBCTDataset(val_inputs, val_targets)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

In [3]:
class DistillationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        #self.logit_alpha = nn.Parameter(torch.tensor(0.0))
        #self.lamda = lamda
        self.hard = nn.MSELoss()
        self.soft = nn.MSELoss()

    def forward(self, student_out, teacher, targets, alpha):
        
        #alpha = torch.sigmoid(self.logit_alpha)
        hard_loss = self.hard(student_out, targets)
        soft_loss = self.soft(student_out, teacher)

        return alpha * soft_loss + (1 - alpha) * hard_loss
    
criterion_distillation = DistillationLoss()
criterion_inference = nn.MSELoss()


In [8]:
model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

HARUnet_modelname = r"HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
teacher_model = torch.load(os.path.join(model_dir,HARUnet_modelname)).to(device)

teacher_model = teacher_model.to(device)

student_model = ResUNet().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    student_model = torch.nn.DataParallel(student_model)
    
optimizer = optim.Adam(student_model.parameters(), lr=1e-3)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=5,
    threshold=1e-4,      # consider changes smaller than this as "no improvement"
    threshold_mode='rel',
    cooldown=0,
    min_lr=1e-10,
    verbose=True
)


C:\Users\au711969\AppData\Local\Temp\ipykernel_872\4082220779.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher_model = torch.load(os.path.join(model_dir,HARUnet_m

Using 3 GPUs!


c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [7]:
early_stopping = EarlyStopping(patience=20, delta=1e-7)
    
train_loss_history = []
train_inpsnr_history = []
train_outpsnr_history = []

#train_issim_history = []
#train_ossim_history = []

val_loss_history = []
val_inpsnr_history = []
val_outpsnr_history = []

#val_issim_history = []
#val_ossim_history = []

teacher_model.eval()
epochs = 1000
for epoch in range(epochs):
    student_model.train()

    train_loss = 0.0
    train_inpsnr = 0
    train_outpsnr = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    alpha = 0.5

    for train_inputs, train_targets in train_loop:
        
        train_inputs, train_targets = train_inputs.unsqueeze(1).to(device), train_targets.unsqueeze(1).to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(train_inputs)

        optimizer.zero_grad()      
        train_outputs = student_model(train_inputs)
        
        
        batch_loss = criterion_distillation(train_outputs, teacher_outputs, train_targets, alpha)
        
        batch_loss.backward()
        
        optimizer.step()  

        train_loss += batch_loss.item()
        train_inpsnr += batch_psnr(train_inputs, train_targets)
        train_outpsnr += batch_psnr(train_outputs, train_targets)

    train_loss /= len(train_loader)
    epoch_train_inpsnr = train_inpsnr / len(train_loader)
    epoch_train_outpsnr = train_outpsnr / len(train_loader)

    train_loss_history.append(train_loss)
    train_inpsnr_history.append(epoch_train_inpsnr)
    train_outpsnr_history.append(epoch_train_outpsnr)

    student_model.eval()
    val_loss = 0.0
    val_inpsnr = 0
    val_outpsnr = 0
    val_BM3Drltvpsnr = 0
    count = 0
    for val_inputs, val_targets in val_loader:

        val_inputs, val_targets = val_inputs.unsqueeze(1).to(device), val_targets.unsqueeze(1).to(device)

        with torch.no_grad():
            val_outputs = student_model(val_inputs)
        if batch_psnr(val_inputs, val_targets) != float('inf'):
            val_loss += criterion_inference(val_outputs, val_targets).item()
            val_inpsnr += batch_psnr(val_inputs, val_targets)
            val_outpsnr += batch_psnr(val_outputs, val_targets)
        else:
            count = count + 1

    val_loss /= (len(val_loader)-count)
    epoch_val_inpsnr = val_inpsnr / (len(val_loader)-count)
    epoch_val_outpsnr = val_outpsnr / (len(val_loader)-count)

    val_loss_history.append(val_loss)
    val_inpsnr_history.append(epoch_val_inpsnr)
    val_outpsnr_history.append(epoch_val_outpsnr)

    train_loop.set_postfix(train_loss=train_loss)
    #if early_stopping.best_loss is None or val_loss < early_stopping.best_loss:
    torch.save(student_model.state_dict(), "DistilledHARUnet_ResUNet_best_0_.pth")

    print(f'Epoch [{epoch+1}/{epochs}], Train loss: {train_loss:.8f}, Input PSNR: {epoch_train_inpsnr:.3f}, Output PSNR: {epoch_train_outpsnr:.3f}, Alpha: {alpha:.3f}') 
    print(f'............, Validation loss: {val_loss:.8f}, Validation Input PSNR: {epoch_val_inpsnr:.3f}, Validation Output PSNR: {epoch_val_outpsnr:.3f}')
    #print(alpha_vals)
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        total_training_epochs = epoch+1
        break

Epoch 1/1000:   0%|          | 0/1564 [00:00<?, ?it/s]c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torch\cuda\nccl.py:16: UserWarning: PyTorch is not compiled with NCCL support
  warnings.warn("PyTorch is not compiled with NCCL support")
Epoch 1/1000: 100%|██████████| 1564/1564 [09:24<00:00,  2.77it/s]


Epoch [1/1000], Train loss: 0.00104224, Input PSNR: 20.531, Output PSNR: 31.474, Alpha: 0.500
............, Validation loss: 0.00063966, Validation Input PSNR: 19.932, Validation Output PSNR: 32.784


Epoch 2/1000: 100%|██████████| 1564/1564 [08:57<00:00,  2.91it/s]


Epoch [2/1000], Train loss: 0.00037441, Input PSNR: 20.532, Output PSNR: 34.624, Alpha: 0.500
............, Validation loss: 0.00044934, Validation Input PSNR: 19.930, Validation Output PSNR: 34.610
Best loss = 0.00044934190267007597


Epoch 3/1000: 100%|██████████| 1564/1564 [08:54<00:00,  2.93it/s]


...

Epoch [36/1000], Train loss: 0.00008507, Input PSNR: 20.531, Output PSNR: 41.082, Alpha: 0.500
............, Validation loss: 0.00031342, Validation Input PSNR: 19.926, Validation Output PSNR: 38.641


Epoch 37/1000: 100%|██████████| 1564/1564 [08:41<00:00,  3.00it/s]


Epoch [37/1000], Train loss: 0.00008327, Input PSNR: 20.531, Output PSNR: 41.180, Alpha: 0.500
............, Validation loss: 0.00031097, Validation Input PSNR: 19.923, Validation Output PSNR: 38.721


Epoch 38/1000: 100%|██████████| 1564/1564 [08:41<00:00,  3.00it/s]


Epoch [38/1000], Train loss: 0.00008619, Input PSNR: 20.531, Output PSNR: 41.031, Alpha: 0.500
............, Validation loss: 0.00031979, Validation Input PSNR: 19.927, Validation Output PSNR: 38.509
Early stopping triggered.


In [8]:
# Assuming 'model' is your trained model
torch.save(student_model, f'DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_at_{epoch+1}epochs_0_.pth')
#model = torch.load("BM3D-LUnet_trainedon_CBCT_CadavarData_at_91epochs_lr001.pth")

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

test_savefilename = os.path.join(savedir,f"Test_CBCT_inputs.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)

test_savefilename = os.path.join(savedir,f"Test_CBCT_targets_bm3d_{sigma}.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)



In [4]:
pickled_test_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_noisyCBCT_patches.pkl"
pickled_test_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_origCBCT_patches.pkl"
test_inputs = load_data(pickled_test_inputs)  # Shape: (N, 1, H, W)
test_targets = load_data(pickled_test_targets)  # Shape: (N, 1, H, W)

test_dataset = CBCTDataset(test_inputs, test_targets)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0,pin_memory=True)

In [ ]:
model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

DistHARU2ResUnet_modelname = r"DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_at_50epochs_.pth"

In [12]:

student_model = torch.load(os.path.join(model_dir,DistHARU2ResUnet_modelname)).to(device)
student_model1 = student_model.half().cuda()
student_model1.eval()
test_loss = 0.0
test_inpsnr = 0
test_outpsnr = 0
test_BM3Drltvpsnr = 0
for test_inputs, test_targets in test_loader:

    test_inputs, test_targets = test_inputs.unsqueeze(1).to(device), test_targets.unsqueeze(1).to(device)
    optimizer.zero_grad()
            
    with torch.no_grad():
        test_outputs = student_model1(test_inputs.half().cuda())
    if batch_psnr(test_inputs, test_targets) != float('inf'):
        test_loss += criterion_inference(test_outputs, test_targets).item()
        test_inpsnr += batch_psnr(test_inputs, test_targets)
        print(f'Batch_psnr = {batch_psnr(test_inputs, test_targets)}, total test_inpsnr = {test_inpsnr}')
        test_outpsnr += batch_psnr(test_outputs, test_targets)

test_loss /= (len(test_loader)-1)
epoch_test_inpsnr = test_inpsnr /(len(test_loader)-1)
epoch_test_outpsnr = test_outpsnr / (len(test_loader)-1)
epoch_test_BM3Drltvpsnr = test_BM3Drltvpsnr / (len(test_loader)-1)
        

print(f'............, Test loss: {test_loss:.8f}, Test Target PSNR: {epoch_test_inpsnr:.3f}, Test outPSNR: {epoch_test_outpsnr:.3f}')
#print(f'Validation loss: {val_loss:.8f}, Validation inPSNR: {epoch_val_inpsnr:.3f}, Validation outPSNR: {epoch_val_outpsnr:.3f}')

C:\Users\au711969\AppData\Local\Temp\ipykernel_872\3930211603.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  student_model = torch.load(os.path.join(model_dir,DistHARU2

Batch_psnr = 19.91053943205366, total test_inpsnr = 19.91053943205366
Batch_psnr = 20.530110242985188, total test_inpsnr = 40.44064967503885
Batch_psnr = 19.692595831123278, total test_inpsnr = 60.133245506162126
Batch_psnr = 19.819907763485027, total test_inpsnr = 79.95315326964715
Batch_psnr = 19.308625665893402, total test_inpsnr = 99.26177893554055
Batch_psnr = 19.493434403256806, total test_inpsnr = 118.75521333879735
Batch_psnr = 19.363000089709, total test_inpsnr = 138.11821342850635
Batch_psnr = 19.90722136042103, total test_inpsnr = 158.02543478892738
Batch_psnr = 20.209012191641044, total test_inpsnr = 178.23444698056844
Batch_psnr = 19.190885865307767, total test_inpsnr = 197.4253328458762
Batch_psnr = 19.377453940759594, total test_inpsnr = 216.8027867866358
Batch_psnr = 20.069220882144524, total test_inpsnr = 236.87200766878033
Batch_psnr = 19.91066036151888, total test_inpsnr = 256.7826680302992
Batch_psnr = 20.994081264225215, total test_inpsnr = 277.7767492945244
Batch_

In [11]:
# Assuming loss_history and psnr_history are your lists of metrics
data = {
    'Epoch': range(1, len(train_loss_history) + 1),  # Start epoch count from 1
    'Training Loss': train_loss_history,
    'Training BM3D PSNR': train_inpsnr_history,
    'Training outPSNR': train_outpsnr_history,
    'Validation Loss': val_loss_history,
    'Validation BM3D PSNR': val_inpsnr_history,
    'Validation outPSNR': val_outpsnr_history,
    'Testing Loss': test_loss,
    'Testing BM3D PSNR': epoch_test_inpsnr,
    'Testing outPSNR': epoch_test_outpsnr,
}

# Create a DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('TrainingnTesting_metrics_propDestilledHARUnet_epochs{epoch}_0_.csv', index=False)  # Set index=False to avoid saving row indices  